# Microenvironment Tutorial (10x)

Huetracerでの10x Visium HD向けマイクロエンバイロメント解析を、設定読み込みから保存まで一通り実行するノートブックです。

## Notebook Overview

This notebook demonstrates the end-to-end microenvironment analysis workflow for 10x Visium HD data in HueTracer.

- Load runtime configuration and initialize analysis environment.
- Prepare parameters and required reference files (NicheNet ligand-target table).
- Estimate microenvironments with VAE and project results to spatial coordinates.
- Optionally refine labels interactively and run downstream differential analyses.
- Save analysis outputs for cell-cell interaction and follow-up tutorials.

## Environment Setup and Imports

In [ ]:
# =========================
# Library Imports
# =========================

import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import json

import matplotlib.pyplot as plt
import seaborn as sns

import bin2cell as b2c

import ipywidgets as widgets
from IPython.display import display, clear_output

# Custom modules
import huetracer
from huetracer import ConfigSelector
from huetracer.widgets import create_nichenet_downloader_widget

# =========================
# Environment Configuration
# =========================

os.environ["NVCC_PREPEND_FLAGS"] = "--std=c++17"
os.environ["CCCL_IGNORE_DEPRECATED_CPP_DIALECT"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ["TF_CPP_MAX_VLOG_LEVEL"] = "0"
warnings.filterwarnings("ignore", message=".*must be within the support of the distribution.*")

# =========================
# Device Setup
# =========================

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
device_str = device.type
print(f"Using device: {device}")

# =========================
# Jupyter and Scanpy Settings
# =========================

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
pd.set_option('display.max_columns', None)
sc.set_figure_params(figsize=[10, 10], dpi=100)

# =========================
# Random Seed / Reproducibility
# =========================

SEED = 42

# Core Python/Numpy RNGs
random.seed(SEED)
np.random.seed(SEED)

# Scanpy global seed (used by stochastic Scanpy routines)
sc.settings.seed = SEED

# Torch RNGs
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Deterministic Torch behavior where possible
# Note: some ops/devices may not have deterministic implementations.
torch.backends.cudnn.benchmark = False
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print(f"Warning: deterministic algorithms could not be fully enabled: {e}")

# PYTHONHASHSEED must be set before interpreter startup for full effect.
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================
# Utility Functions
# =========================

def clear_mem():
    """Clear memory by running garbage collection and CUDA cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =========================
# VisiumHD config file selector
# =========================

selector = ConfigSelector(base_dir_default="/app/data/input")
selector.display()

# Loaded settings are available via selector.last_config
# e.g. cfg = selector.last_config

In [ ]:
# Runtime config binding (after clicking "Load")
if selector.last_config is None:
    raise RuntimeError("Please click 'Load' in the widget first.")

globals().update(selector.last_config)

## Set Parameters

解析対象領域、対象遺伝子、近傍セル数、下流解析で使うグループ設定をここで定義します。

In [ ]:
# =========================
# Parameters to be input
# =========================

# Species-dependent target genes used in histogram examples
Species = "Human"
# Species = "Mouse"

if Species == "Human":
    target_genes = ["TNFSF11"]
else:
    target_genes = ["Col1a1"]

# Target cell type for DEG / volcano analysis
target_cell_type = annotation_dict["C1"]
# target_cell_type = "Airway epithelial cells (CAPN8+, ELF3+)"

# Definition of neighborhood cells for microenvironment modeling
neighbor_cell_numbers = 19

# Volcano plot groups
group1_environments = ["0", "2"]
group2_environments = ["1", "3"]

# =========================
# Lasso Selection image quality
# =========================
# Background image downsample factor for the interactive relabeling widgets.
#   1.0  = full resolution  (high quality, very heavy)
#   0.5  = half resolution  (good quality, moderate)
#   0.25 = quarter resolution (lightweight, recommended default)
#   0.1  = 1/10 resolution  (minimal, very fast)
lasso_downsample_factor = 0.25

# Output filename used in "Save Outputs"
h5ad_sc_microenvironment_full_filename = SAMPLE_NAME + "_single_cell_microenvironment.h5ad"
h5ad_microenvironment_full_save_path = os.path.join(RESULTS_PATH, h5ad_sc_microenvironment_full_filename)

## Download and Prepare NicheNet Ligand-Target CSV

細胞間相互作用解析で利用するligand-target辞書を、必要に応じてダウンロードして利用可能な変数に登録します。

In [ ]:
# =========================
# Interactive widgets for CCI-related file preparation
# =========================

nichenet = huetracer.create_nichenet_downloader_widget(
    base_dir=BASE_DIR, runtime_namespace=globals()
)
nichenet.display()

## Microenvironment Estimation with VAE

核セグメンテーション・細胞種アノテーション済みの AnnData を読み込み、各細胞の近傍遺伝子発現パターンを Variational Autoencoder (VAE) で学習することで、空間的なマイクロエンバイロメントクラスターを推定します。

In [ ]:
# =========================
# Microenvironment estimation with variational autoencoder.
# Gene expression data of neighbor cells around each center cell is used as input.
# =========================

# --- Step 1: Load nucleus-segmented AnnData and retrieve spatial library ID ---
h5ad_predicted_full_save_path = os.path.join(RESULTS_PATH, SAMPLE_NAME + "_nucleus_predicted.h5ad")
h5ad_save_path = os.path.join(RESULTS_PATH, SAMPLE_NAME + "_b2c.h5ad")
sp_adata_predicted = sc.read_h5ad(h5ad_predicted_full_save_path)
sp_adata_raw = sc.read_h5ad(h5ad_save_path)
lib_id = list(sp_adata_raw.uns['spatial'].keys())[0]

# --- Step 2: Apply spatial mask to restrict analysis to the ROI ---
cell_mask = ((sp_adata_predicted.obs['array_row'] >= mask_x1_val) & 
             (sp_adata_predicted.obs['array_row'] <= mask_x2_val) & 
             (sp_adata_predicted.obs['array_col'] >= mask_y1_val) & 
             (sp_adata_predicted.obs['array_col'] <= mask_y2_val)
            )
sp_adata_microenvironment = sp_adata_predicted[cell_mask].copy()
del sp_adata_predicted, cell_mask; clear_mem()

# --- Step 3: Select informative genes (DEG union HVG) for VAE input ---

# 3-1. Count cells per cell type to identify cell types with too few cells
group_counts = sp_adata_microenvironment.obs['predicted_cell_type'].value_counts()
valid_groups = group_counts[group_counts > 1].index.tolist()

# 3-2. Exclude cell types with only 1 cell (required for Wilcoxon DEG); normalize and log-transform
mask = sp_adata_microenvironment.obs["predicted_cell_type"].isin(valid_groups)
filtered_adata = sp_adata_microenvironment[mask].copy()
filtered_adata.X = filtered_adata.layers["counts"].copy()
sc.pp.normalize_total(filtered_adata, target_sum = 1e6)
sc.pp.log1p(filtered_adata)

filtered_adata.raw = None

# 3-3. Run Wilcoxon DEG test per cell type and collect top-100 marker genes
sc.tl.rank_genes_groups(
    filtered_adata,
    groupby='predicted_cell_type',
    method='wilcoxon',
    n_genes=100,
    use_raw=False
)

# Handle both structured recarray (old scanpy) and plain ndarray (new scanpy)
names = filtered_adata.uns["rank_genes_groups"]["names"]

if isinstance(names, np.ndarray) and names.dtype.names is not None:
    arr = np.vstack([names[g] for g in names.dtype.names]).T
else:
    arr = np.asarray(names)

top_genes = np.unique(arr.astype(str).ravel())
top_genes = top_genes[top_genes != "nan"]

common_hvg = [g for g in top_genes.tolist() if g in filtered_adata.var_names]

# 3-4. Select top-100 highly variable genes (Seurat v3) and take union with DEG genes
sc.pp.highly_variable_genes(filtered_adata, n_top_genes=100, layer="counts", flavor='seurat_v3')
ref_hvg_100 = filtered_adata.var[filtered_adata.var['highly_variable']].index.tolist()
all_genes = set(common_hvg) | set(ref_hvg_100)
# Sort for deterministic feature order across runs/environments.
final_gene_list = sorted([g for g in all_genes if g in sp_adata_microenvironment.var_names])
del filtered_adata; clear_mem()

# --- Step 4: Normalize, subset to final gene list, and min-max scale for VAE input ---
sc.pp.normalize_total(sp_adata_microenvironment, target_sum = 1e0)
sp_adata_microenvironment = sp_adata_microenvironment[:, final_gene_list].copy()
coords = sp_adata_microenvironment.obs[["array_row", "array_col"]].values
X = sp_adata_microenvironment.X.toarray() if hasattr(sp_adata_microenvironment.X, "toarray") else sp_adata_microenvironment.X
data_min = X.min()
data_max = X.max()
X = (X - data_min) / (data_max - data_min)
cell_types = sp_adata_microenvironment.obs['predicted_cell_type']
print("highly variable genes:", len(final_gene_list))

# --- Step 5: Build neighborhood microenvironment tensors (k nearest cells concatenated) ---
print(f"Device: {device}")
analyzer = huetracer.SpatialMicroenvironmentAnalyzer(coords, X, k_neighbors=neighbor_cell_numbers, device=device)
indices, microenv_data = analyzer.build_microenvironment_data()
del coords, X, indices, microenv_data; clear_mem()

# --- Step 6: Train VAE on neighborhood tensors and extract latent features ---
vae_model = analyzer.train_vae(latent_dim=32, epochs=1000, batch_size=16384, lr=4e-4, dim_1=128, dim_2=128, weight_decay=1e-4)
analyzer.extract_latent_features()
del vae_model; clear_mem()

# --- Step 7: Cluster latent space with UMAP + Leiden and visualize results ---
umap_embedding, clusters = analyzer.perform_umap_clustering(
    cell_type_data=cell_types,
    seed=SEED,
    clustering_backend="cpu",
    missing_vertex_policy="neighbor",
)
analyzer.visualize_results()
analyzer.visualize_scanpy_results()
huetracer.plot_all_clusters_highlights(analyzer)
huetracer.plot_all_cell_type_highlights(analyzer)

# --- Step 8: Assign microenvironment cluster labels and overlay on tissue image ---
sp_adata_microenvironment.obs['predicted_microenvironment'] = analyzer.adata.obs['leiden'].astype(str).to_numpy()
sp_adata_microenvironment.obs['predicted_microenvironment'] = sp_adata_microenvironment.obs['predicted_microenvironment'].astype("category")
huetracer.create_hires_overlay_plot(sp_adata_microenvironment, lib_id, SAMPLE_NAME, RESULTS_PATH, file_name="cell_type", color_key="predicted_cell_type")
huetracer.create_hires_overlay_plot(sp_adata_microenvironment, lib_id, SAMPLE_NAME, RESULTS_PATH, file_name="microenvironment", color_key="predicted_microenvironment", TITLE="MicroEnv")
del umap_embedding; clear_mem()

In [ ]:
# Store original labels for reset / safety reference
predicted_microenvironment_original = analyzer.adata.obs['leiden'].values.astype(str)
predicted_cell_type_original = sp_adata_microenvironment.obs["predicted_cell_type"].values.astype(str)

# Ensure categorical dtype consistency for downstream analyses
sp_adata_microenvironment.obs['predicted_microenvironment'] = sp_adata_microenvironment.obs['predicted_microenvironment'].astype('category')
sp_adata_microenvironment.obs['predicted_cell_type'] = sp_adata_microenvironment.obs['predicted_cell_type'].astype('category')


### Save Checkpoint (before Interactive Relabeling)

VAE推定まで（GPU向き処理）が終わった時点で、以降の工程を再開できるようにチェックポイントを保存します。  
この後の Interactive Relabeling / Downstream Analysis はCPUインスタンスでも実行しやすいため、ここでインスタンスを切り替えて再開できます。

In [ ]:
# =========================
# Save Checkpoint
# =========================
# VAE解析完了後、Interactive Relabeling前にチェックポイントを保存します。
# 保存後にカーネルを再起動し、下の「Load Checkpoint」セルから再開できます。

_ckpt_dir = os.path.join(RESULTS_PATH, "checkpoint_before_relabeling")
os.makedirs(_ckpt_dir, exist_ok=True)

# AnnData (sp_adata_microenvironment)
sp_adata_microenvironment.write_h5ad(os.path.join(_ckpt_dir, "sp_adata_microenvironment.h5ad"))


# NumPy arrays
np.save(os.path.join(_ckpt_dir, "predicted_microenvironment_original.npy"), predicted_microenvironment_original)
np.save(os.path.join(_ckpt_dir, "predicted_cell_type_original.npy"), predicted_cell_type_original)

# Metadata (lib_id, h, w)
with open(os.path.join(_ckpt_dir, "meta.json"), "w") as _f:
    json.dump({"lib_id": lib_id, "h": int(h), "w": int(w)}, _f)

print(f"✅ Checkpoint saved to: {_ckpt_dir}")
print("  - sp_adata_microenvironment.h5ad")
print("  - merged.parquet")
print("  - predicted_microenvironment_original.npy")
print("  - predicted_cell_type_original.npy")
print("  - meta.json")


## Optional: Interactive Relabeling

> Optional step. Use this section if you want to manually refine predicted microenvironment or cell-type labels.

> このセクションはCPU環境でも実行可能です。GPUでVAE推定後にインスタンスを切り替えて作業を続ける運用を想定しています。

> Recommended JupyterLab extensions:
- jupyter-matplotlib
- jupyter-widgets-jupyterlab-manager
- jupyterlab-plotly

How to use:
1. Select microenvironment groups to display.
2. Choose selection mode (Lasso/Rectangle).
3. Draw region(s) on the image.
4. Apply selection and update labels.
5. Use zoom as needed.
6. Re-plot to confirm updates.

### Resume from Checkpoint

カーネル再起動後、またはGPU環境からCPU環境へ切り替えた後はここから再開できます。  
下のセルで保存済みの変数を復元します。  
実行前に「Environment Setup」「Set Parameters」は先に実行してください。

In [ ]:
# =========================
# Load Checkpoint
# =========================
# カーネル再起動後にこのセルを実行して変数を復元します。
# 事前に「Environment Setup」「Set Parameters」セクションを実行しておいてください。

_ckpt_dir = os.path.join(RESULTS_PATH, "checkpoint_before_relabeling")

# AnnData (sp_adata_microenvironment)
sp_adata_microenvironment = sc.read_h5ad(os.path.join(_ckpt_dir, "sp_adata_microenvironment.h5ad"))


# NumPy arrays
predicted_microenvironment_original = np.load(
    os.path.join(_ckpt_dir, "predicted_microenvironment_original.npy"), allow_pickle=True
)
predicted_cell_type_original = np.load(
    os.path.join(_ckpt_dir, "predicted_cell_type_original.npy"), allow_pickle=True
)

# Metadata
with open(os.path.join(_ckpt_dir, "meta.json")) as _f:
    _meta = json.load(_f)
lib_id = _meta["lib_id"]
h = _meta["h"]
w = _meta["w"]

# sp_adata_raw (Downstream Analysis で必要)
h5ad_save_path = os.path.join(RESULTS_PATH, SAMPLE_NAME + "_b2c.h5ad")
if os.path.exists(h5ad_save_path):
    sp_adata_raw = sc.read_h5ad(h5ad_save_path)
    print(f"✅ sp_adata_raw loaded from: {h5ad_save_path}")
else:
    print(f"⚠️  sp_adata_raw not found at: {h5ad_save_path}")

print(f"✅ Checkpoint loaded from: {_ckpt_dir}")
print(f"   sp_adata_microenvironment: {sp_adata_microenvironment.shape}")

### Pre-check: Spatial distribution of predicted microenvironment and cell type

The following two cells visualize the current spatial distribution of `predicted_microenvironment` and `predicted_cell_type` before any modification.  
Use these plots to review the initial clustering and cell type predictions.

In [ ]:
# =========================
# Check microenvironment clusters before modification
# =========================

fig = huetracer.plot_spatial_plotly_fast(
    sp_adata_microenvironment,
    color_col="predicted_microenvironment",
    basis="spatial_cropped_150_buffer",
    img_key="0.5_mpp_150_buffer",
    point_size=3,
    point_opacity=0.6,
    title="Predicted microenvironment"
)

# # If you want to see the original data...
# sc.pl.spatial(
#     sp_adata_microenvironment, color='predicted_microenvironment',
#     title='Predicted microenvironment',
#     size=20,
#     alpha_img=0.2,
#     img_key="0.5_mpp_150_buffer", basis="spatial_cropped_150_buffer",
#     legend_fontsize=5,
#     groups=None,
#     spot_size=1,
#     frameon=False
# )

In [ ]:
# Release resources and remove the Plotly widget from memory after use
del fig
gc.collect()

In [ ]:
# =========================
# Check Predicted Cell Type before modification
# =========================

fig = huetracer.plot_spatial_plotly_fast(
    sp_adata_microenvironment,
    color_col="predicted_cell_type",
    basis="spatial_cropped_150_buffer",
    img_key="0.5_mpp_150_buffer",
    point_size=3,
    point_opacity=0.6,
    title="Predicted Cell Type"
)

# # If you want to see the original data...
# sc.pl.spatial(
#     sp_adata_microenvironment, color='predicted_cell_type',
#     title='Predicted predicted_cell_type',
#     size=20,
#     alpha_img=0.2,
#     img_key="0.5_mpp_150_buffer", basis="spatial_cropped_150_buffer",
#     legend_fontsize=5,
#     groups=None,
#     spot_size=1,
#     frameon=False
# )

In [ ]:
# Release resources and remove the Plotly widget from memory after use
del fig
gc.collect()

### Manual lasso-based microenvironment relabeling

This widget allows you to manually relabel microenvironment clusters by lasso-selecting regions on the plot.  
You can apply changes, update the AnnData object, and export results as CSV using the provided buttons.

In [ ]:
# =========================
# Microenvironment modification
# =========================

selector = huetracer.lasso_selection_microenvironment(
    sp_adata=sp_adata_microenvironment,
    lib_id=lib_id,
    clusters='predicted_microenvironment',
    basis='spatial_cropped_150_buffer', # or 'spatial'
    img_key='0.5_mpp_150_buffer', # or 'lowres'/'hires'
    downsample_factor=0.05, # or lasso_downsample_factor
)


In [ ]:
# Release resources and remove the Plotly widget from memory after use
selector.fig.close()

### Distance-based microenvironment relabeling

This widget enables automatic relabeling of microenvironment clusters based on spatial distance to selected cell types or clusters.  
You can choose the distance method (nearest, centroid, knn_mean, distance_transform) and set thresholds to assign new labels according to spatial proximity.

In [ ]:
# =========================
# Distance-based microenvironment modification (example)
# =========================

selector = huetracer.distance_selection_microenvironment(
    sp_adata=sp_adata_microenvironment,
    lib_id=lib_id,
    clusters='predicted_microenvironment',
    cell_types='predicted_cell_type',
    basis='spatial_cropped_150_buffer',
    img_key='0.5_mpp_150_buffer',
    downsample_factor=0.05, # or lasso_downsample_factor
)


In [ ]:
# Release resources and remove the Plotly widget from memory after use
selector.fig.close()

### Manual lasso-based cell type relabeling

This widget allows you to manually relabel cell type clusters by lasso-selecting regions on the plot.  
You can apply changes, update the AnnData object, and export results as CSV using the provided buttons, just like with microenvironment relabeling.

In [ ]:
# =========================
# Cell type modification
# =========================

selector = huetracer.lasso_selection_cell_type(
    sp_adata=sp_adata_microenvironment,
    lib_id=lib_id,
    clusters='predicted_microenvironment',
    cell_types='predicted_cell_type',
    basis='spatial_cropped_150_buffer',
    img_key='0.5_mpp_150_buffer',
    downsample_factor=0.05, # or lasso_downsample_factor,
)

In [ ]:
# Release resources and remove the Plotly widget from memory after use
selector.fig.close()

In [ ]:
# Reset modification
# 
# sp_adata_microenvironment.obs['predicted_microenvironment'] = predicted_microenvironment_original
# sp_adata_microenvironment.obs['predicted_cell_type'] = predicted_cell_type_original


## Downstream Analysis and Visualization

このセクションは `sp_adata_raw` と `sp_adata_microenvironment` が利用可能であることを前提に実行します。  
チェックポイント復元後にそのまま実行できます。

In [ ]:
# Gene expression and microenvironment
plt.close('all')
%matplotlib widget
huetracer.plot.create_spatial_widget(
    sp_adata_raw, 
    sp_adata_microenvironment
)

In [ ]:
# Gene expression difference among clusters (recommended workflow)
plt.close('all')
%matplotlib inline
deg_results_df = huetracer.plot.plot_deg_by_microenvironment(
    sp_adata_raw=sp_adata_raw,
    sp_adata_microenvironment=sp_adata_microenvironment,
    target_cell_type=target_cell_type,
    n_genes=8, # トップ8遺伝子を表示
    save=True,  # プロットをPDFで保存
    save_path_for_today=RESULTS_PATH
)

In [ ]:
# Volcano plot of gene expression between clusters
volcano_df = huetracer.plot.plot_volcano_between_microenvironments(
    sp_adata_raw=sp_adata_raw,
    sp_adata_microenvironment=sp_adata_microenvironment,
    target_cell_type=target_cell_type,
    group1_environments=group1_environments,
    group2_environments=group2_environments,
    save=True,
    save_path_for_today=RESULTS_PATH,
    sample_name=SAMPLE_NAME,
)

### Gene Expression Histograms

`target_genes` に指定した遺伝子の発現量分布をグリッド形式で可視化します。  
`show_nonzero_only=False` で全細胞（ゼロ含む）、`True` で発現細胞のみの分布を表示します。

In [ ]:
# Gene expression histograms for target genes

# microenvironmentが推定済みの細胞のみを対象にする
common_cells = sp_adata_microenvironment.obs_names.intersection(sp_adata_raw.obs_names)

# 全発現量（ゼロ含む）
fig = huetracer.plot.plot_gene_histograms_batch(
    adata=sp_adata_raw[common_cells],
    target_genes=target_genes,
    n_cols=4,
    bins=100,
    log_scale=True,
    show_nonzero_only=False,
)
plt.show()

# 非ゼロ発現量のみ
fig = huetracer.plot.plot_gene_histograms_batch(
    adata=sp_adata_raw[common_cells],
    target_genes=target_genes,
    n_cols=4,
    bins=100,
    log_scale=True,
    show_nonzero_only=True,
)
plt.show()

### Report Plots

推定した microenvironment ラベルの空間散布図と、microenvironmentごとの細胞種構成比ヒートマップを出力します。

In [ ]:
# Spatial scatter plot + cell-type composition heatmap
huetracer.plot.create_report_plots(
    sp_adata_microenvironment=sp_adata_microenvironment,
    lib_id=lib_id,
    sample_name=SAMPLE_NAME,
    save_path=RESULTS_PATH,
    basis="spatial_cropped_150_buffer",
    img_key="0.5_mpp_150_buffer",
)


## Save Outputs

推定済みmicroenvironmentをh5adとして保存し、必要な設定値をJSONに書き出して後続解析（CCIなど）に引き継ぎます。

In [ ]:
# Save spatial data for cell-cell interaction analysis
sp_adata_microenvironment.write_h5ad(h5ad_microenvironment_full_save_path)

# Update config (merge additional runtime parameters into existing config file)
from huetracer.widgets import update_config_file

print(f"Config target: {B2C_CONFIG_SAVE_PATH}")

# Collect optional keys safely from current runtime
candidate_keys = [
    "SAMPLE_NAME",
    "BASE_DIR",
    "source_image_path",
    "SOURCE_IMAGE_PATH",
    "sc_filtered_path",
    "EXPRESSION_PATH",
    "EXPRESSION_PATH_8UM",
    "RESULTS_PATH",
    "DATE",
    "TMP_PATH",
    "B2C_CONFIG_SAVE_PATH",
    "annotation_dict",
    "file_nichenet",
    "h5ad_microenvironment_full_save_path",
    "mask_x1_val",
    "mask_x2_val",
    "mask_y1_val",
    "mask_y2_val",
]

add_config = {}
for k in candidate_keys:
    if k in globals():
        add_config[k] = globals()[k]

update_config_file(
    B2C_CONFIG_SAVE_PATH, add_config, merge=True
)

print(f"💾 Merged {len(add_config)} config fields into {B2C_CONFIG_SAVE_PATH}")
print("Done! Go ahead.")